# Step 4: CLAP on dog barks (EmotionalCanines)

Same question as step 1, now for dogs: does a small classifier on CLAP embeddings beat CLAP zero-shot?

- Data: Emotional Canines (Dang et al., ACM MM 2025), github.com/tmdang1101/EmotionalCanines, licence CC BY-NC-SA 4.0 (non-commercial; we only download it, never redistribute).
  1,400 barks from 199 dogs, Siberian Husky and Shiba Inu, 700 each. Two labels per bark, three levels each:
  **arousal** (Low, Medium, High) and **valence** (Negative, Neutral, Positive).
- Split: the dataset's own train/test split, which keeps every dog on one side (600/100 per breed). There is no per-dog ID, so C is chosen by 5-fold CV on train (folds may share dogs, which only affects the choice of C, not the test score).
- Features: CLAP audio embeddings, same as step 1 (PyTorch; step 2 showed the phone's ONNX gives identical results).
- Chance for 3 classes is about 0.33 macro F1.

In [ ]:
import os, glob, json, time, subprocess
import numpy as np, pandas as pd, librosa, torch, transformers
from transformers import ClapModel, ClapProcessor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
SR = 48000
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| transformers", transformers.__version__)

## 1. Download (Git LFS)

In [ ]:
if not os.path.isdir("EmotionalCanines"):
    subprocess.run("git lfs install && git clone --depth 1 https://github.com/tmdang1101/EmotionalCanines", shell=True, check=True)
rows = []
for breed in ["husky", "shiba"]:
    for split in ["train", "test"]:
        df = pd.read_csv(f"EmotionalCanines/{breed}_{split}_labels.csv")
        df["breed"], df["split"] = breed, split
        df["path"] = [f"EmotionalCanines/{breed}/{split}/{a}.wav" for a in df.audio_id]
        rows.append(df)
meta = pd.concat(rows, ignore_index=True)
missing = [p for p in meta.path if not os.path.exists(p) or os.path.getsize(p) < 1000]
print(len(meta), "labelled barks;", len(missing), "missing or LFS pointers")
print(meta.groupby(["breed", "split"]).size())

## 2. Audio and CLAP embeddings

In [ ]:
audio = [librosa.load(p, sr=SR, mono=True)[0] for p in meta.path]
d = np.array([len(a) / SR for a in audio])
print(f"clip length: median {np.median(d):.2f} s, min {d.min():.2f} s, max {d.max():.2f} s")

model = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device).eval()
proc = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
def embed(batch):
    inp = proc.feature_extractor(batch, sampling_rate=SR, padding="repeatpad", return_tensors="pt")
    with torch.no_grad():
        e = model.audio_projection(model.audio_model(input_features=inp["input_features"].to(device)).pooler_output)
    return torch.nn.functional.normalize(e, dim=-1).cpu().numpy()
t0 = time.time()
X = np.concatenate([embed(audio[i:i+16]) for i in range(0, len(audio), 16)])
print("embeddings:", X.shape, f"in {time.time()-t0:.0f} s")

## 3. Zero-shot prompts

In [ ]:
PROMPTS = {
    "arousal": {"Low": "a calm, relaxed dog barking softly",
                "Medium": "a dog barking",
                "High": "a very excited, agitated dog barking loudly"},
    "valence": {"Negative": "an angry, aggressive or distressed dog barking",
                "Neutral": "a dog barking",
                "Positive": "a happy, playful dog barking"},
}
def text_emb(texts):
    tok = proc.tokenizer(texts, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        t = model.text_projection(model.text_model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"]).pooler_output)
    return torch.nn.functional.normalize(t, dim=-1).cpu().numpy()

## 4. Score on the test dogs

In [ ]:
results, cms = {}, {}
for task, classes in [("arousal", ["Low", "Medium", "High"]), ("valence", ["Negative", "Neutral", "Positive"])]:
    y = np.array([classes.index(v) for v in meta[task]])
    T = text_emb([PROMPTS[task][c] for c in classes])
    zs = (X @ T.T).argmax(1)
    for breed in ["husky", "shiba", "both"]:
        sel = np.ones(len(meta), bool) if breed == "both" else (meta.breed == breed).values
        tr = sel & (meta.split == "train").values
        te = sel & (meta.split == "test").values
        grid = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
                            {"logisticregression__C": [0.001, 0.01, 0.1, 1]}, scoring="f1_macro",
                            cv=StratifiedKFold(5, shuffle=True, random_state=42)).fit(X[tr], y[tr])
        pred = grid.predict(X[te])
        maj = np.full(te.sum(), np.bincount(y[tr]).argmax())
        r = {"test_barks": int(te.sum()),
             "majority_f1": round(f1_score(y[te], maj, average="macro"), 3),
             "zero_shot_f1": round(f1_score(y[te], zs[te], average="macro"), 3),
             "logreg_f1": round(f1_score(y[te], pred, average="macro"), 3),
             "logreg_acc": round(accuracy_score(y[te], pred), 3),
             "C": grid.best_params_["logisticregression__C"]}
        results[f"{task}_{breed}"] = r
        cms[f"{task}_{breed}"] = confusion_matrix(y[te], pred).tolist()
        print(f"{task:8s} {breed:6s} majority {r['majority_f1']:.3f} | zero-shot {r['zero_shot_f1']:.3f} | CLAP+LR {r['logreg_f1']:.3f} (acc {r['logreg_acc']:.3f}, C={r['C']})")
print("\nconfusion matrices (rows true, cols predicted, class order as above):")
for k in ["arousal_both", "valence_both"]: print(k, cms[k])
print("\nRESULTS_JSON", json.dumps({"results": results, "confusion": cms}))